![Platform+](images/banner.png)

# LAB 5: APPLY AI TO TEXT
_Hands-on Snowflake for data practitioners_

> **Persona track: Platform Admin** — same lab, your angle.

👉 **In this lesson, we'll build AI-powered guardrails: redact PII, moderate content, and monitor Cortex AI consumption.**

📚 **By the end you'll be able to:**
- Redact PII from text columns automatically
- Filter content by semantic conditions
- Compare cost/quality tradeoffs between AI models
- Query Cortex AI usage from ACCOUNT_USAGE
- Build views that enforce AI-based governance transparently

**Time:** ~30 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

> 📌 **Why you can call these AI functions:** access to Cortex AI functions is granted through the [`SNOWFLAKE.CORTEX_USER`](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access) **database role**, already granted to your role during setup. A **database role** is a role scoped to a single database (here, the built-in `SNOWFLAKE` database); it can't be activated directly in a session, so it's *granted to* an account role like `{{my_role}}`, which inherits its privileges. As the admin, you'll govern exactly this role — and which models it can reach — yourself in **§7**. Everything you build lands in your own database.

## Snowflake AI Functions — the big picture
Snowflake gives you a whole **family of AI functions** (part of *Cortex AISQL*) that run large language models for you, right inside the database. The magic: **nobody has to change how they work** — no model training, no GPUs, no data leaving Snowflake. Anyone with access calls them like any other function in **SQL or Python**. For a Platform Admin, that's exactly *why governance matters*: powerful AI is one `GRANT` away, so you set the guardrails.

There are many of these functions. In this lab you'll use these as your governance toolkit:

| Function | What it does |
|---|---|
| `AI_REDACT` | Masks PII (names, emails, phone numbers) in text |
| `AI_FILTER` | A plain-English boolean test — semantic content moderation |
| `AI_COMPLETE` | Generates text; used here to compare model cost vs quality |
| `AI_COUNT_TOKENS` | Estimates token cost before running at scale |

👉 Browse the full list any time in the [Cortex AI Functions docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql).

## 1 · The Problem: PII in Free Text
Customer feedback often contains personally identifiable information (PII) — names, emails, phone numbers embedded in comments. Before analysts can run AI on this data, you need to ensure PII is stripped.

Let's see what's in the feedback.

In [ ]:
-- COMMENT is a reserved word in Snowflake, so we alias it to CUSTOMER_COMMENT for a clean result
SELECT feedback_id, comment AS customer_comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL;


> 🔎 **Review the output.** Scan the comments. Do any mention names, contact details, or other personal info? Even if a sample looks clean, you can't assume all rows are safe — that's why you automate redaction.

## 2 · Redact PII With AI_REDACT
[`AI_REDACT`](https://docs.snowflake.com/en/sql-reference/functions/ai_redact) detects and masks PII entities (names, emails, phone numbers, addresses), replacing them with placeholders like `[NAME]` or `[EMAIL_ADDRESS]`. It's **non-destructive** — the source table is untouched, so authorized roles still see the original.

We seeded a few feedback rows (ids 300–309) with PII so you can see it work.

In [ ]:
SELECT feedback_id,
       LEFT(comment, 80) AS original_comment,
       LEFT(AI_REDACT(comment), 80) AS redacted_comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE feedback_id BETWEEN 300 AND 309;

> 🔎 **Review the output.** Where AI_REDACT found PII, you'll see placeholders; where it found nothing sensitive, the text passes through unchanged. The original data stays intact in the source table.

## 3 · Content Moderation With AI_FILTER
Beyond *masking* PII, some comments simply **shouldn't be surfaced as-is** — they single out a specific employee by name or expose someone's personal contact details, so they don't belong in a shared business report even when the wording is perfectly polite.

[`AI_FILTER`](https://docs.snowflake.com/en/sql-reference/functions/ai_filter) evaluates text against a natural-language condition and returns TRUE/FALSE. Use `PROMPT('… {0}', column)` to feed a column into the condition — think of it as a programmable WHERE clause that understands meaning.

In [ ]:
-- Which comments are safe to publish in a business report as-is?
-- We sample the seeded rows (300-309, which name people / expose contact details)
-- alongside ordinary comments so you can see the filter split them.
SELECT feedback_id,
       LEFT(comment, 80) AS comment_preview,
       AI_FILTER(PROMPT('This comment can be published in a business report as-is, without exposing personal contact details or singling out a specific named employee: {0}', comment)) AS publishable
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE feedback_id BETWEEN 300 AND 309
   OR feedback_id BETWEEN 100 AND 115
ORDER BY publishable;

> 📌 **Note:** The condition is plain English, not regex — you can write any boolean test: `'This comment names a specific employee'`, `'This feedback is a complaint, not praise'`, `'This text is relevant to coffee or food service'`. AI_FILTER judges *meaning*, so a comment that exposes a phone number or singles out a barista fails the publish test **even though the words are perfectly polite** — that's the difference from a keyword rule.

### Use AI_FILTER to find only actionable feedback.
🧩 **Your turn.** AI_FILTER's power is that its condition is **plain English** — *you* write the test. In the cell below the condition is blanked out. Replace `______` with a natural-language statement that is TRUE only when a comment contains a specific, actionable suggestion for improvement — the `: {0}` that injects the comment is already there, so you only write the condition text. Then run it. Stuck? Reveal the reference below the cell.

In [ ]:
-- Only show feedback that contains a specific, actionable suggestion
SELECT feedback_id, score, comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL
  AND AI_FILTER(PROMPT('______: {0}', comment))   -- 🧩 write just the plain-English condition
LIMIT 10;

> <details><summary>🧩 Stuck? Reference solution</summary>
>
>     This feedback contains a specific, actionable suggestion for improvement
>
> That text goes in place of `______`, leaving `PROMPT('This feedback contains a specific, actionable suggestion for improvement: {0}', comment)`. Your wording can differ — AI_FILTER judges meaning, not exact words.
> </details>

## 4 · Model Selection: Cost vs. Quality
AI functions bill per **token** — a token is a chunk of text the model reads, very roughly **~4 characters of English**. Snowflake meters tokens and converts them to **credits** at a *per-model* rate (the same credits you costed out in Lab 4) — so the model you approve directly drives spend.

Your team will ask *'which model should we use?'* As gatekeeper, you set the approved list and balance quality against cost. Run the same prompt through a small (8b) and a large (70b) model and compare.

In [ ]:
-- Compare two models on the same task
WITH feedback_sample AS (
    SELECT comment FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
    WHERE comment IS NOT NULL LIMIT 1
)
SELECT 'llama3.1-8b' AS model,
       AI_COMPLETE('llama3.1-8b', 'Summarize this customer feedback in one sentence: ' || comment) AS response,
       AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', comment) AS input_tokens
FROM feedback_sample
UNION ALL
SELECT 'llama3.1-70b',
       AI_COMPLETE('llama3.1-70b', 'Summarize this customer feedback in one sentence: ' || comment),
       AI_COUNT_TOKENS('ai_complete', 'llama3.1-70b', comment)
FROM feedback_sample;

> 📌 **Note:** **Admin guidance:** small models (8b) for classification/sentiment/formatting — faster, cheaper; large models (70b+) for nuanced summarization/reasoning. Token count is the same across models; the **credit rate per token** is what differs. For simple tasks the 8b often matches the 70b — so test before defaulting to a bigger (pricier) model.

### Estimate a job's cost before you run it
You just saw that the *model* sets the credit rate per token. The other half of the bill is **how many tokens** a job will process. Before running an AI function across a whole table, size it up with `AI_COUNT_TOKENS` — it counts tokens **without** calling the model, so it's essentially free to check. Here we estimate running `AI_REDACT` over every feedback comment.

In [ ]:
-- Estimate the token cost of running AI_REDACT across all feedback
SELECT COUNT(*) AS total_rows,
       SUM(AI_COUNT_TOKENS('ai_redact', comment)) AS total_tokens,
       ROUND(AVG(AI_COUNT_TOKENS('ai_redact', comment)), 1) AS avg_tokens_per_row
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
WHERE comment IS NOT NULL;

## 5 · Monitor AI Consumption
Cortex AI usage shows up in `SNOWFLAKE.ACCOUNT_USAGE`. This is how you answer *'how much are we spending on AI?'* and *'who is the heaviest user?'*

In [ ]:
-- Cortex AI credit usage by day (may have up to 2-hour latency)
SELECT usage_date, service_type, SUM(credits_used) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE (service_type ILIKE 'AI_%' OR service_type ILIKE '%CORTEX%')
  AND usage_date >= DATEADD('day', -7, CURRENT_DATE())
GROUP BY usage_date, service_type
ORDER BY usage_date DESC;

> ⚠️ ACCOUNT_USAGE views have up to 2 hours of latency. If you just ran AI functions for the first time this session, you may not see that activity yet. In production, build alerts on these.

In [ ]:
-- Who is running AI functions? (last 7 days)
SELECT user_name,
       COUNT(*) AS ai_query_count,
       SUM(credits_used_cloud_services) AS cloud_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE query_text ILIKE '%AI_%(%'
  AND start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY user_name
ORDER BY ai_query_count DESC
LIMIT 10;

## 6 · Build a Governed View
The deliverable: a **governed view** that applies `AI_REDACT` and `AI_FILTER` inline — `AI_REDACT` masks personal details in the text it keeps, while `AI_FILTER` drops rows that shouldn't be surfaced at all (ones built around a named individual or their contact details). Grant analysts the view (not the base table) and the governance is baked in.

> 💡 **Why a view, not a table?** You could freeze AI output into a *table* to avoid re-paying token cost — but governance wants the opposite tradeoff. A **view** re-applies `AI_REDACT` / `AI_FILTER` on every query, so consumers always get the *latest* source rows with protection guaranteed — nothing slips through on a stale snapshot. The cost is real (AI runs per query), which is exactly what the note below addresses.

In [ ]:
CREATE OR REPLACE VIEW {{my_db}}.SANDBOX.FEEDBACK_SAFE AS
SELECT
    f.feedback_id,
    f.store_id,
    f.customer_id,
    f.score,
    AI_REDACT(f.comment) AS comment,   -- PII removed at query time
    f.feedback_date
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
WHERE f.comment IS NOT NULL
  AND AI_FILTER(PROMPT('This comment can be published in a business report as-is, without exposing personal contact details or singling out a specific named employee: {0}', f.comment));  -- content gate: drop rows unsafe to surface

In [ ]:
-- Test the governed view: PII redacted, unpublishable rows filtered out
SELECT * FROM {{my_db}}.SANDBOX.FEEDBACK_SAFE LIMIT 5;

> 📌 **Note:** For high-volume tables, running AI_REDACT/AI_FILTER on every query adds latency and cost. Pre-compute the redacted text on a schedule (Task or Dynamic Table) and point the view at the pre-computed column.

## 7 · Govern AI Access and Model Usage
As a Platform Admin, you control **who** can use AI functions and **which models** they can access — critical for cost control, compliance, and preventing misuse. Two levers: database roles (who) and model RBAC application roles (which models).

> ⚠️ The statements below require `ACCOUNTADMIN`/`SECURITYADMIN` and are shown **for reference** — you'd run them in an admin session, not as a student.

### Lever 1 · Database roles for AI access
| Database role | What it grants |
|---|---|
| `SNOWFLAKE.CORTEX_USER` | All Cortex AI functions (AI_COMPLETE, AI_SENTIMENT, AI_REDACT, AI_FILTER, …) |
| `SNOWFLAKE.CORTEX_EMBED_USER` | AI_EMBED only — for teams that only need embeddings |

A role also needs the account-level `USE AI FUNCTIONS` privilege (granted to PUBLIC by default).

In [ ]:
-- REFERENCE ONLY — requires SECURITYADMIN or ACCOUNTADMIN. Role names are examples.
-- Full AI access:
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE DATA_ENGINEER_RL;
-- Embedding-only (cheaper, limited scope):
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER TO ROLE SEARCH_SERVICE_RL;
-- Remove AI access entirely:
-- REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE INTERN_RL;

### Lever 2 · Restrict available models (Model RBAC)
By default, every role has access to **all** Cortex models — the `CORTEX-MODEL-ROLE-ALL` application role is automatically granted through `PUBLIC`. To restrict which models a role can use, revoke the blanket role and grant only the specific `CORTEX-MODEL-ROLE-<model>` roles each team needs.

> ⚠️ To revoke `CORTEX-MODEL-ROLE-ALL` from PUBLIC, use `SNOWFLAKE.LOCAL.REVOKE_FROM_PUBLIC_APPLICATION_ROLE` — a plain `REVOKE APPLICATION ROLE` won't work for the bootstrap grant.

In [ ]:
-- REFERENCE ONLY — requires ACCOUNTADMIN.
-- Step 1: Revoke blanket model access from PUBLIC (uses special procedure):
-- CALL SNOWFLAKE.LOCAL.REVOKE_FROM_PUBLIC_APPLICATION_ROLE('APP_ROLE', 'CORTEX-MODEL-ROLE-ALL');
-- Step 2: Grant specific model roles to the roles that need them:
-- GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-LLAMA3.1-8B" TO ROLE DATA_ENGINEER_RL;
-- GRANT APPLICATION ROLE SNOWFLAKE."CORTEX-MODEL-ROLE-SNOWFLAKE-ARCTIC-EMBED-M-V1.5" TO ROLE SEARCH_SERVICE_RL;
-- Check which model roles are granted to PUBLIC:
-- SHOW GRANTS TO ROLE PUBLIC;

### Putting it together: practical governance scenarios
| Scenario | Action |
|---|---|
| Analyst needs AI for sentiment | Grant `SNOWFLAKE.CORTEX_USER` to their role |
| Search service only needs embeddings | Grant `SNOWFLAKE.CORTEX_EMBED_USER` (not full CORTEX_USER) |
| Intern should not use AI at all | Don't grant any CORTEX database role |
| Team keeps using an expensive 405b model | Revoke `CORTEX-MODEL-ROLE-ALL` from PUBLIC, grant only the models you approve |
| Runaway overnight batch costs | ACCOUNT_USAGE alerts (Section 5) + restrict model roles |

> 💡 **Least privilege:** default to `CORTEX_EMBED_USER`, escalate to `CORTEX_USER` only when a team shows a valid generative use case.

---
## List the available Cortex models
As the platform admin, a natural first question is: **what AI models can users in this account actually reach?** It's the raw material for every choice in §4 and §7 — you can't approve or restrict a list you haven't seen.

Listing the account's Cortex base models normally needs account-level visibility a data role doesn't (and shouldn't) have. So this lab exposes it through an **owner's-rights stored procedure** — `PLATPLUS_DB.PUBLIC.LIST_CORTEX_MODELS()` — owned by `ACCOUNTADMIN`. You call it with your everyday role and it returns the current list on your behalf, without granting you standing account-wide privileges. (Same pattern as the resource-monitor helpers in Lab 4.)

In [ ]:
-- Owner's-rights proc (owned by ACCOUNTADMIN) returns the Cortex base models
-- available in this account — name + type — without granting you account-level access.
CALL PLATPLUS_DB.PUBLIC.LIST_CORTEX_MODELS();

> 🔎 **Review the output.** Each row is a Cortex base model you can name in an AI function (e.g. the model argument to `AI_COMPLETE`). `MODEL_TYPE = CORTEX_BASE` marks Snowflake's hosted foundation models. This is the raw list you'd govern with **Model RBAC** application roles (Lever 2 above) — you can't restrict what you haven't enumerated.

With the model landscape in view, let's put those functions to work. This is where CoCo really earns its keep: describe the policy you need, let it map that to the right function(s), then help you write the SQL.

Your security team asks: *"Give analysts a feed of the feedback with all personal data stripped and anything inappropriate held back."*

That's **two** guardrails — remove PII, and gate on content. Which functions enforce each? Don't guess — ask CoCo.

**Step 1 — ask CoCo which functions fit:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I need to give analysts a version of PLATPLUS_DB.ROASTERY_SALES.FEEDBACK where personal information is removed from the COMMENT text and any inappropriate comments are held back. Which Snowflake Cortex AI functions enforce each of those two guardrails, and why? SQL-callable functions only.</pre>

> 🔎 **Read CoCo's recommendation.** It should point you to **`AI_REDACT`** to strip PII and **`AI_FILTER`** to gate on content. Now that you know the functions, ask CoCo to build the governed object.

**Step 2 — ask CoCo to build it:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Write a CREATE OR REPLACE VIEW in my database (schema SANDBOX, view FEEDBACK_ANALYST_SAFE) over PLATPLUS_DB.ROASTERY_SALES.FEEDBACK that redacts PII from the comment with AI_REDACT and keeps only rows AI_FILTER judges appropriate for a business report. Only rows where comment IS NOT NULL.</pre>

Review CoCo's output, paste it below, and run it.

In [ ]:
-- Paste CoCo's Step 2 query on the blank line below, then run it


### Iterate — tune the filter 💡
Run it and scan what got through. Too strict (dropping useful feedback) or too loose (letting noise in)? AI_FILTER quality lives in the wording of its condition. Ask CoCo:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">My AI_FILTER condition for "appropriate for a business report" is dropping useful comments (or letting noise through). Suggest a sharper natural-language condition, and explain the tradeoff.</pre>

### More ways to lean on CoCo with AI functions 🤖
CoCo isn't just for writing queries. A few prompts worth keeping in your back pocket:
- **Monitor spend:** *"Show Cortex AI credit usage by model over the last 7 days from ACCOUNT_USAGE."*
- **Right-size a model:** *"For PII redaction at scale, which model is cheapest that still does the job?"*
- **Pre-compute:** *"Turn this governed view into a scheduled pre-computed table to cut query-time AI cost."*

🤖 **Try it now.** Pick one of the prompts above (or invent your own) and paste it into CoCo. Review what it suggests, then paste the SQL into the cell below and run it.

> 📌 CoCo's suggestion is a starting point — you may need to iterate. Refine based on the result, just like you would with a colleague.

In [ ]:
-- Paste CoCo's suggestion on the blank line below, then run it


## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ('SANDBOX.FEEDBACK_SAFE governed view exists', f"SELECT COUNT(*) FROM {my_db}.INFORMATION_SCHEMA.VIEWS WHERE table_schema='SANDBOX' AND table_name='FEEDBACK_SAFE'", 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import platform_admin_quiz as quiz
quiz.display_question("q3")

### Key takeaways. ❄️
- AI_REDACT removes PII from text — use it before sharing data with analysts
- AI_FILTER is a semantic WHERE clause — use it for content moderation and relevance
- Model selection matters: smaller models for simple tasks, larger for nuanced work
- ACCOUNT_USAGE tracks AI consumption — set up monitoring and cost alerts
- Governed views apply AI inline so consumers get clean data without knowing the mechanics
- Pre-compute expensive AI results when query-time cost or latency is unacceptable
- Database roles (CORTEX_USER / CORTEX_EMBED_USER) control who can use AI; Model RBAC application roles restrict which models — default to least privilege

**The Other Personas:**
- The **Data Analyst** notebook scores sentiment, classifies issues, and rolls up feedback summaries with AI_AGG
- The **Data Engineer** notebook builds an AI enrichment pipeline — standardize, extract, translate, embed — then operationalizes it with a Dynamic Table

> :bulb: Each persona lab is independent — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.